# EE 4XX - Lab 1

[TODO: Team member names here]

### Imports

In [1]:
import cv2
import numpy as np
import glob
import time
from sympy import symbols, cos, sin, Matrix
from pymycobot.mycobot import MyCobot

## Task 1: Camera Detection
To be able to detect and range objects properly, the distortion introduced by the camera must be known.  One model is the pinhole camera model, in which radial and tangential distortion can be calculated through a camera's intrinsic parameters.  These intrinsics allows us to undistort the image, which is necessary for being able to accurately detect and range objects.

### 1.1 - Capture calibration photos
One method for calibration is to calculate distortion based on photos of a known pattern, such as a checkerboard.  To do this, the corners lying inside the pattern are detected from the image (see https://www.mathworks.com/help/vision/ug/calibration-patterns.html for a visualization).  Since we know that the lines on the checkerboard are parallel and we know the physical dimensions of the squares on the checkerboard, we can find out how much the camera distorts the image.

In order to accurately calculate the camera intrinsics, multiple images (10 or more) should be taken, with the checkerboard appearing in different areas of the image and at different angles.  One possible way to capture the minimum of 10 photos is:
1. Checkerboard centered
2. Top-left tilted
3. Top-right tilted
4. Bottom-left tilted
5. Bottom-right tilted
6. Close distance
7. Medium distance
8. Far distance
9. Rotated left
10. Rotated right

Additionally, the physical checkerboard itself should be undistorted (e.g., attached to a flat surface, such as a clipboard) to avoid introducing additional distortion on top of the camera's intrinsic distortion. Finally, make sure the camera and the checkerboard are stationary when taking the photo to avoid motion blur.



In [17]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# TODO: How many points on each axis of the checkerboard?
dim_1 = 10
dim_2 = 7

# Prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(9,6,0)
# TODO: How many object points should be stored for a n x m checkerboard? 
#       Change the first dimension in (None, 3) to this value.
objp = np.zeros((dim_1 * dim_2, 3), np.float32)
objp[:,:2] = np.mgrid[0:dim_1, 0:dim_2].T.reshape(-1,2)

# Compensate for square size (measure physically)
square_size = 0.022
objp *= square_size
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3D point in real world space
imgpoints = [] # 2D points in image plane.

In [ ]:
# TODO: Run the provided code and capture images using the camera.
# Check that the points are detected correctly before saving! 
#       To capture an image: press space (' ')
#       To save an image: press 's'
#       To retake an image: press 'r'
#       To quit: press 'q'

cap = cv2.VideoCapture(0)
img_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    display = frame.copy()
    cv2.imshow("Calibration", display)

    key = cv2.waitKey(1) & 0xFF

    if key == ord(' '):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(gray, (dim_1, dim_2), None)

        if found:
            candidate = frame.copy()

            corners2 = cv2.cornerSubPix(
                gray, corners, (11, 11), (-1, -1), criteria
            )

            cv2.drawChessboardCorners(
                candidate, (dim_1, dim_2), corners2, found
            )

            cv2.imshow("Candidate - press s to save, r to retake", candidate)

            candidate_key = cv2.waitKey(0) & 0xFF

            if candidate_key == ord('s'):
                filename = f"images/calib_{img_idx}.jpg"
                cv2.imwrite(filename, frame)
                print(f"Saved {filename}")
                img_idx += 1

            elif candidate_key == ord('r'):
                print("Retaking image.")

            cv2.destroyWindow("Candidate - press s to save, r to retake")

        else:
            print("Checkerboard not detected, try again.")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Saved images/calib_0.jpg
Retaking image.
Retaking image.
Saved images/calib_1.jpg


### 1.2 - Calculate and save camera intrinsics

Now that the images are captured, we need to process them and save the camera intrinsics to a file (this way calibration only has to be run once).

In [ ]:
images = glob.glob('images/*.jpg')

print(len(images))

for fname in images:
    img = cv2.imread(fname)

    # Convert image to grayscale for processing
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (dim_1, dim_2), None)
 
    # If found, add object points, image points (after refining them)
    # TODO: What do we append to objpoints, imgpoints?
    if ret:
        objpoints.append(objp)

        # Location of corners in image
        corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

# TODO: First three arguments for
#       cv2.calibrateCamera(objectPoints, imagePoints, imageSize, None, None) 
#       (last two are None)
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

# TODO: Save calibration, distortion matrices to .csv for future use
np.savetxt('calib_mtx.csv', None, delimiter=',')
np.savetxt('calib_dist.csv', None, delimiter=',')

11


Check that the undistortion works (run provided code)

In [ ]:
img = cv2.imread(images[0])
h, w = img.shape[:2]
newcammtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

undistort = False
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    if (undistort):
        frame = cv2.undistort(frame, mtx, dist, None, newcammtx)
        # x, y, w, h = roi
        # frame = frame[]
    
    cv2.imshow("Camera", frame)

    key = cv2.waitKey(1)

    if key == ord('u'):
        undistort = not undistort
    elif key == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()
cv2.waitKey(1)

### 1.3 - Verify marker detection and ranging